# 01 Initial load (run once)
Full copy of the source `sales` table into Bronze, builds the first Silver snapshot and sets the watermark.

In [ ]:
def jdbc_options():
    """Read connection details from a Databricks secret scope (never hard-code credentials)."""
    scope = "sales_pipeline"
    host = dbutils.secrets.get(scope, "pg_host")
    database = dbutils.secrets.get(scope, "pg_database")
    return {
        "url": f"jdbc:postgresql://{host}:5432/{database}?sslmode=require",
        "user": dbutils.secrets.get(scope, "pg_user"),
        "password": dbutils.secrets.get(scope, "pg_password"),
        "driver": "org.postgresql.Driver",
    }

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.read.format("jdbc")
    .options(**jdbc_options())
    .option("dbtable", "sales")
    .load()
    .withColumn("ingestion_time", F.current_timestamp())
)
df.write.format("delta").mode("overwrite").saveAsTable("bronze.sales")

In [ ]:
# First Silver snapshot: latest version of every customer.
spark.sql("""
INSERT OVERWRITE silver.sales
SELECT customer_id, amount, region, updated_at
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
    FROM bronze.sales
)
WHERE rn = 1
""")

In [ ]:
max_ts = spark.table("bronze.sales").agg(F.max("updated_at")).collect()[0][0]
if max_ts is not None:
    spark.sql(f"""
    UPDATE metadata.pipeline_metadata
    SET watermark = TIMESTAMP'{max_ts}'
    WHERE pipeline_name = 'sales_pipeline'
    """)
print(f"Watermark set to {max_ts}")